# Bhagavad Gita KG — Theme Layer (C1)

Layers a curated flat `Theme` taxonomy onto the graph, linking verses to themes
deterministically by reusing the `Term` layer.

- **Prerequisite:** run `gita_kg.ipynb` first (this notebook layers on top of the
  already-loaded v1 graph; it does not re-parse verses).
- Deterministic + idempotent: re-running rebuilds the theme layer with no duplicates.

## 1. Imports & connect

In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


ROOT = find_repo_root(Path.cwd())
PKG = ROOT / "gita-knowledge-graph"
sys.path.insert(0, str(PKG))

import gita_kg as gk

load_dotenv(PKG / ".env", override=True)
cfg = gk.load_config()
driver = GraphDatabase.driver(cfg.uri, auth=(cfg.user, cfg.password))
driver.verify_connectivity()
print("connected:", cfg.uri, "->", cfg.database)

connected: bolt://localhost:7687 -> neo4j


## 2. Load the theme layer (constraint → themes)

In [2]:
def run_ops(ops):
    with driver.session(database=cfg.database) as session:
        for cypher, params in ops:
            session.run(cypher, **params)


run_ops(gk.theme_constraint_ops())
run_ops(gk.theme_ops())
print("themes loaded")

themes loaded


## 3. Coverage report (verses per theme)

In [3]:
with driver.session(database=cfg.database) as s:
    rows = s.run(
        "MATCH (th:Theme) "
        "OPTIONAL MATCH (v:Verse)-[:MENTIONS_THEME]->(th) "
        "RETURN th.name AS theme, th.category AS category, count(v) AS verses "
        "ORDER BY verses DESC"
    ).values()
for theme, category, verses in rows:
    flag = "  <EMPTY>" if verses == 0 else ""
    print(f"{verses:4d}  {theme:20s} [{category}]{flag}")

 191  jnana                [metaphysics]
 156  samsara              [metaphysics]
 152  karma                [ethics]
 142  senses-mind          [psychology]
 138  atman                [metaphysics]
 119  detachment           [ethics]
  95  bhakti               [devotion]
  80  yoga                 [path]
  77  guna                 [metaphysics]
  64  sacrifice-austerity  [ritual]
  47  dharma               [ethics]
  15  moksha               [metaphysics]
  10  brahman              [metaphysics]


## 4. Verification

In [4]:
def val(cypher):
    with driver.session(database=cfg.database) as s:
        return s.run(cypher).values()


print("themes:", val("MATCH (th:Theme) RETURN count(th)")[0][0])
print(
    "top karma verses:",
    val(
        "MATCH (v:Verse)-[r:MENTIONS_THEME]->(:Theme {name:'karma'}) "
        "RETURN v.id AS id, r.weight AS w ORDER BY w DESC LIMIT 5"
    ),
)
print(
    "themes of 2.47:",
    val(
        "MATCH (:Verse {id:'2.47'})-[:MENTIONS_THEME]->(th) "
        "RETURN th.name ORDER BY th.name"
    ),
)

themes: 13
top karma verses: [['4.14', 4], ['5.14', 4], ['2.47', 4], ['3.35', 4], ['6.1', 4]]
themes of 2.47: [['detachment'], ['karma']]


## 5. Close

In [5]:
driver.close()